# Assignment: Real-Time Intrusion Detection — Task 1 (Dataset Exploration and Feature Understanding)

This notebook explores the Network Intrusion Dataset (Kaggle — ISCX/CIC-style CSV flows) available in this project folder. It:

- Loads the 8 CSV files and reports shapes and columns.
- Lists all features (union across files) and identifies common features.
- Infers a data type (numeric/categorical/boolean-like) for each feature.
- Provides a clear description and a short justification of relevance for distinguishing normal vs. attack traffic (feature-wise or via justified feature groups).
- Concludes with a short summary of packet-/flow-level vs. derived/aggregated features.

This notebook is designed to run top-to-bottom without external dependencies beyond pandas/numpy.


In [ ]:
# Imports and display settings
import pandas as pd
import numpy as np
from collections import Counter
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

# Dataset files (present in the repo root)
filenames = [
    'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
    'Friday-WorkingHours-Morning.pcap_ISCX.csv',
    'Monday-WorkingHours.pcap_ISCX.csv',
    'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
    'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
    'Tuesday-WorkingHours.pcap_ISCX.csv',
    'Wednesday-workingHours.pcap_ISCX.csv',
]

# Verify all files exist
for f in filenames:
    assert Path(f).exists(), f'Missing dataset file: {f}'

def normalize_col(name: str) -> str:
    if not isinstance(name, str):
        return str(name)
    # Strip leading/trailing spaces and collapse internal whitespace; keep original casing/words
    s = ' '.join(name.strip().split())
    return s

dfs = []
original_columns_per_file = []
for file in filenames:
    # 'Infinity' and similar strings appear in some files; treat them as NaN for numeric inference
    df = pd.read_csv(file, skipinitialspace=True, low_memory=False, na_values=['Infinity', 'inf', 'Inf'])
    original_columns_per_file.append(list(df.columns))
    # Normalize columns for alignment across files
    df.rename(columns={c: normalize_col(c) for c in df.columns}, inplace=True)
    dfs.append(df)

print('Loaded files and shapes:')
for i, (file, df) in enumerate(zip(filenames, dfs), start=1):
    print(f"{i}. {file} -> shape={df.shape}")


## Task 1.1 — List All Features

We report columns per file (as found) and compute the union and the intersection of feature names across the 8 CSV files.


In [ ]:
# Columns per file (after normalization)
for i, (file, df) in enumerate(zip(filenames, dfs), start=1):
    print(f"\nFile {i}: {file}")
    print(f"Columns ({len(df.columns)}):")
    print(list(df.columns))

# Union and intersection of columns
all_cols = []
for df in dfs:
    all_cols.extend(list(df.columns))
col_presence = Counter(all_cols)
union_cols = sorted(col_presence.keys())
common_cols = sorted(set(dfs[0].columns).intersection(*[set(df.columns) for df in dfs[1:]]))

print(f"\nTotal unique features (union): {len(union_cols)}")
print(union_cols)
print(f"\nCommon features present in all files (intersection): {len(common_cols)}")
print(common_cols)

# Presence (how many files each feature appears in)
presence_df = pd.DataFrame({
    'feature': union_cols,
    'present_in_files': [col_presence[c] for c in union_cols]
}).sort_values(['present_in_files','feature'], ascending=[False, True]).reset_index(drop=True)
display(presence_df.head(20))


## Task 1.2 — Feature Data Types, Descriptions, and Relevance

We infer a coarse data type for each feature (numeric/boolean-like/categorical) and generate a description and a relevance justification.

Descriptions and justifications are derived from (a) a curated mapping for CICFlowMeter-style features and (b) heuristics based on feature name patterns (e.g., `IAT`, `Packets/s`, `Flag Count`).


In [ ]:
# Helper: infer coarse dtype for a feature across all files
def infer_feature_dtype(name: str) -> str:
    series_list = []
    for df in dfs:
        if name in df.columns:
            series_list.append(df[name])
    if not series_list:
        return 'unknown'
    s = pd.concat(series_list, ignore_index=True)
    # Try numeric coercion
    s_num = pd.to_numeric(s, errors='coerce')
    non_na = s.notna().sum()
    numeric_ratio = s_num.notna().sum() / max(1, non_na)
    if numeric_ratio >= 0.95:
        # Distinguish bool-like from general numeric
        uniq = pd.Series(s_num.dropna().round(6).unique())
        try:
            uniq_int = set(uniq.astype(int))
        except Exception:
            uniq_int = set()
        if len(uniq) <= 3 and uniq_int.issubset({0,1,2}):
            return 'boolean-like (0/1 or small discrete)'
        # integer-like vs float-like
        if (s_num.dropna() % 1 == 0).mean() > 0.98:
            return 'numeric (integer-like)'
        return 'numeric (float-like)'
    # Otherwise, categorical / text
    return 'categorical/text'

# Curated short descriptions for common CIC/ISCX flow features
desc_map = {
    'Destination Port': 'TCP/UDP destination port number for the flow.',
    'Source Port': 'TCP/UDP source port number for the flow.',
    'Flow Duration': 'Total duration of the bidirectional flow (typically in milliseconds or microseconds depending on generator).',
    'Total Fwd Packets': 'Total number of packets sent in the forward direction (source to destination).',
    'Total Backward Packets': 'Total number of packets sent in the backward direction (destination to source).',
    'Total Length of Fwd Packets': 'Sum of payload sizes for forward packets (bytes).',
    'Total Length of Bwd Packets': 'Sum of payload sizes for backward packets (bytes).',
    'Fwd Packet Length Max': 'Maximum forward packet length within the flow.',
    'Fwd Packet Length Min': 'Minimum forward packet length within the flow.',
    'Fwd Packet Length Mean': 'Mean forward packet length within the flow.',
    'Fwd Packet Length Std': 'Standard deviation of forward packet lengths.',
    'Bwd Packet Length Max': 'Maximum backward packet length within the flow.',
    'Bwd Packet Length Min': 'Minimum backward packet length within the flow.',
    'Bwd Packet Length Mean': 'Mean backward packet length within the flow.',
    'Bwd Packet Length Std': 'Standard deviation of backward packet lengths.',
    'Flow Bytes/s': 'Throughput: total bytes per second within the flow.',
    'Flow Packets/s': 'Packet rate: total packets per second within the flow.',
    'Flow IAT Mean': 'Mean inter-arrival time between consecutive packets of the flow.',
    'Flow IAT Std': 'Standard deviation of inter-arrival time within the flow.',
    'Flow IAT Max': 'Maximum inter-arrival time between packets.',
    'Flow IAT Min': 'Minimum inter-arrival time between packets.',
    'Fwd IAT Total': 'Total inter-arrival time for forward packets.',
    'Fwd IAT Mean': 'Mean inter-arrival time for forward packets.',
    'Fwd IAT Std': 'Standard deviation of inter-arrival times for forward packets.',
    'Fwd IAT Max': 'Maximum inter-arrival time for forward packets.',
    'Fwd IAT Min': 'Minimum inter-arrival time for forward packets.',
    'Bwd IAT Total': 'Total inter-arrival time for backward packets.',
    'Bwd IAT Mean': 'Mean inter-arrival time for backward packets.',
    'Bwd IAT Std': 'Standard deviation of inter-arrival times for backward packets.',
    'Bwd IAT Max': 'Maximum inter-arrival time for backward packets.',
    'Bwd IAT Min': 'Minimum inter-arrival time for backward packets.',
    'Fwd PSH Flags': 'Count of PSH flag occurrences in forward direction.',
    'Bwd PSH Flags': 'Count of PSH flag occurrences in backward direction.',
    'Fwd URG Flags': 'Count of URG flag occurrences in forward direction.',
    'Bwd URG Flags': 'Count of URG flag occurrences in backward direction.',
    'Fwd Header Length': 'Total size of headers for forward packets (bytes).',
    'Bwd Header Length': 'Total size of headers for backward packets (bytes).',
    'Fwd Packets/s': 'Forward packet rate (packets per second).',
    'Bwd Packets/s': 'Backward packet rate (packets per second).',
    'Min Packet Length': 'Minimum packet length observed in the flow.',
    'Max Packet Length': 'Maximum packet length observed in the flow.',
    'Packet Length Mean': 'Mean packet length within the flow.',
    'Packet Length Std': 'Standard deviation of packet lengths.',
    'Packet Length Variance': 'Variance of packet lengths.',
    'FIN Flag Count': 'Count of FIN flags seen in the flow.',
    'SYN Flag Count': 'Count of SYN flags seen in the flow.',
    'RST Flag Count': 'Count of RST flags seen in the flow.',
    'PSH Flag Count': 'Count of PSH flags seen in the flow.',
    'ACK Flag Count': 'Count of ACK flags seen in the flow.',
    'URG Flag Count': 'Count of URG flags seen in the flow.',
    'CWE Flag Count': 'Count of congestion window reduced (CWR/CWE) flags.',
    'ECE Flag Count': 'Count of ECN-Echo flags.',
    'Down/Up Ratio': 'Download-to-upload packets/bytes ratio for the flow.',
    'Average Packet Size': 'Average size per packet over the flow (bytes).',
    'Avg Fwd Segment Size': 'Average TCP segment size in forward direction.',
    'Avg Bwd Segment Size': 'Average TCP segment size in backward direction.',
    'Fwd Avg Bytes/Bulk': 'Average forward bulk bytes (grouped burst metric).',
    'Fwd Avg Packets/Bulk': 'Average forward bulk packets per burst.',
    'Fwd Avg Bulk Rate': 'Average rate of forward bulks (bursts) per second.',
    'Bwd Avg Bytes/Bulk': 'Average backward bulk bytes (grouped burst metric).',
    'Bwd Avg Packets/Bulk': 'Average backward bulk packets per burst.',
    'Bwd Avg Bulk Rate': 'Average rate of backward bulks (bursts) per second.',
    'Subflow Fwd Packets': 'Packets in the largest identified forward subflow.',
    'Subflow Fwd Bytes': 'Bytes in the largest identified forward subflow.',
    'Subflow Bwd Packets': 'Packets in the largest identified backward subflow.',
    'Subflow Bwd Bytes': 'Bytes in the largest identified backward subflow.',
    'Init_Win_bytes_forward': 'Initial TCP window size advertised by forward endpoint (bytes).',
    'Init_Win_bytes_backward': 'Initial TCP window size advertised by backward endpoint (bytes).',
    'act_data_pkt_fwd': 'Number of forward packets with at least one byte of payload.',
    'min_seg_size_forward': 'Minimum TCP segment size observed in the forward direction.',
    'Active Mean': 'Mean duration of active periods within the flow (data transfer).',
    'Active Std': 'Std of active period durations.',
    'Active Max': 'Max active period duration.',
    'Active Min': 'Min active period duration.',
    'Idle Mean': 'Mean duration of idle (no packets) periods within the flow.',
    'Idle Std': 'Std of idle period durations.',
    'Idle Max': 'Max idle period duration.',
    'Idle Min': 'Min idle period duration.',
    'Label': 'Ground-truth class label indicating BENIGN or a specific attack type.'
}

# Heuristic description and relevance generator
def feature_description_and_relevance(name: str):
    n = name
    if n in desc_map:
        desc = desc_map[n]
    else:
        # Heuristic descriptions based on common tokens
        if 'Port' in n:
            desc = 'TCP/UDP port number involved in the flow.'
        elif 'IAT' in n:
            desc = 'Inter-arrival time metric (timing between packets) for flow/forward/backward.'
        elif 'Bytes/s' in n or 'Packts/s' in n or 'Packets/s' in n:
            desc = 'Rate-based metric (bytes or packets per second) within the flow.'
        elif 'Packet Length' in n:
            desc = 'Statistic of packet length distribution within the flow.'
        elif 'Header Length' in n:
            desc = 'Aggregate size of protocol headers over packets in a direction.'
        elif 'Flag' in n or 'Flags' in n:
            desc = 'TCP flag occurrence/count metric within the flow.'
        elif 'Segment Size' in n:
            desc = 'Average TCP segment size in a direction.'
        elif 'Subflow' in n:
            desc = 'Packets/bytes in a subflow (split of the main flow by heuristics).'
        elif 'Bulk' in n:
            desc = 'Burst/bulk transfer metric summarizing grouped packet bursts.'
        elif 'Active' in n:
            desc = 'Active period (data transfer) duration statistic within the flow.'
        elif 'Idle' in n:
            desc = 'Idle (no traffic) period duration statistic within the flow.'
        elif 'Down/Up Ratio' in n:
            desc = 'Ratio of downstream to upstream traffic intensity.'
        elif 'Duration' in n:
            desc = 'Duration/time-related metric for the flow.'
        elif 'Average Packet Size' in n:
            desc = 'Average size per packet over the entire flow.'
        else:
            desc = 'Flow-level feature generated by a flow-meter tool (CICFlowMeter-like).'
    # Relevance grouping
    low_n = n.lower()
    if 'port' in low_n:
        rel = 'Ports can signal specific services or scans; unusual/rare ports and rapid port changes are indicative of scans or attacks.'
    elif 'iat' in low_n or 'active' in low_n or 'idle' in low_n:
        rel = 'Timing patterns (inter-arrival, active/idle periods) often shift under DoS/port scans/botnets; anomalies in timing variance, max/min are strong signals.'
    elif 'bytes/s' in low_n or 'packets/s' in low_n or 'rate' in low_n:
        rel = 'Throughput or packet rates tend to spike in DoS/scan events or drop for evasive low-and-slow behaviors; rates are strong indicators.'
    elif 'packet length' in low_n or 'average packet size' in low_n or 'segment size' in low_n:
        rel = 'Packet/segment size distributions differ between benign apps and attacks (e.g., many tiny SYNs in SYN floods, large exfiltration chunks).'
    elif 'flag' in low_n:
        rel = 'TCP flag counts (SYN/ACK/FIN/RST/PSH/URG/ECE/CWR) capture handshake anomalies, scans and resets common in intrusions.'
    elif 'header length' in low_n or 'win_bytes' in low_n or 'window' in low_n:
        rel = 'Protocol/header/window metrics can deviate for malicious tools or misconfigured stacks, indicating abnormal flows.'
    elif 'subflow' in low_n or 'bulk' in low_n:
        rel = 'Burstiness and subflow composition reflect behavior patterns; attacks often show distinct burst/group transfer signatures.'
    elif 'length of fwd packets' in low_n or 'length of bwd packets' in low_n or 'total fwd packets' in low_n or 'total backward packets' in low_n:
        rel = 'Unbalanced forward/backward volumes and totals can reveal scans (few responses), exfiltration (more uploads), or DoS (excess requests).'
    elif n == 'Label':
        rel = 'Supervised learning target used for training/evaluation; not used as an input feature for detection.'
    else:
        rel = 'Flow summary metric; deviations from typical application patterns can indicate intrusion activity.'
    return desc, rel

# Build the feature catalog
catalog_rows = []
for feat in union_cols:
    dtype = infer_feature_dtype(feat)
    desc, rel = feature_description_and_relevance(feat)
    catalog_rows.append({
        'feature': feat,
        'in_files': col_presence[feat],
        'in_all_files': feat in common_cols,
        'inferred_dtype': dtype,
        'description': desc,
        'relevance_justification': rel,
    })
feature_catalog = pd.DataFrame(catalog_rows).sort_values(
    ['in_all_files','in_files','feature'], ascending=[False, False, True]
).reset_index(drop=True)
display(feature_catalog.head(20))


## Quick sanity checks

We examine: (a) basic statistics for numeric features; (b) class label distribution, if present.

Note: Some CSVs contain `NaN` from `Infinity` values encountered in original files; this is expected and handled in type inference (numeric coercion).


In [ ]:
# Merge only common columns (to safely concatenate across all files)
dfs_trimmed = [df[common_cols].copy() for df in dfs]
full_df = pd.concat(dfs_trimmed, ignore_index=True)
print('Final concatenated dataset (common columns) shape:', full_df.shape)

# Label distribution if Label is present
if 'Label' in full_df.columns:
    print('\nLabel distribution:')
    print(full_df['Label'].value_counts(dropna=False))

# Identify numeric-like columns by convertibility ratio
def is_numeric_series(s: pd.Series, threshold: float = 0.95) -> bool:
    s_num = pd.to_numeric(s, errors='coerce')
    non_na = s.notna().sum()
    ratio = s_num.notna().sum() / max(1, non_na)
    return ratio >= threshold

numeric_cols = [c for c in full_df.columns if is_numeric_series(full_df[c])]
print(f"\nNumeric columns detected: {len(numeric_cols)} (showing up to 10)")
print(numeric_cols[:10])

if numeric_cols:
    numeric_df = full_df[numeric_cols].apply(pd.to_numeric, errors='coerce')
    display(numeric_df.describe().T.head(12))


## Task 1.3 — Summary: Packet-/Flow-level vs. Derived/Aggregated Features

Most columns in this dataset come from a flow meter (CICFlowMeter-style) that aggregates raw packets into bidirectional flows. Features like `Destination Port`, TCP flag counts (`SYN Flag Count`, `ACK Flag Count`), `Packet Length` statistics, and inter-arrival time (`IAT`) stats reflect packet-level properties summarized at the flow level.

Rate metrics (`Flow Bytes/s`, `Flow Packets/s`, `Fwd/Bwd Packets/s`) and period metrics (`Active*`, `Idle*`) are derived/aggregated statistics computed across the lifetime of a flow rather than a single packet. Bulk/subflow metrics and window sizes are higher-level derived features that capture behavior patterns (burstiness, congestion/windowing) rather than raw packet fields. Overall, the dataset primarily consists of flow-level aggregations of packet attributes, with a smaller set directly tied to canonical packet header fields (ports, flags, header lengths).


## Task 2 — Identify Scapy-Computable Features (Linux vs. Raspberry Pi)

Goal: From the dataset feature list, select a compact subset that can be extracted in real time via Scapy by sniffing packets and keeping lightweight per‑flow state. For each feature we note feasibility on a Linux server and a Raspberry Pi, add a short reason, and include a tiny validation snippet idea. Snippets are strings only — nothing is executed.

Assumptions:
- Per-flow key is a 5‑tuple: `(src, dst, sport, dport, proto)`.
- “Fwd” direction is defined by the first packet observed for that flow; reverse packets are “Bwd”.
- Simple rates (pkts/s, bytes/s) and IAT means are maintained online with O(1) updates per packet.


In [ ]:
# Build a compact feasibility table with validation snippet ideas (strings only; not executed)
rows = []

def mk(feature, linux_yes, pi_yes, reason, snippet):
    rows.append({
        'Feature': feature,
        'Computable on Linux?': 'yes' if linux_yes else 'no',
        'Computable on Raspberry Pi?': 'yes' if pi_yes else 'no',
        'Reason': reason,
        'Validation snippet': snippet.strip()
    })

sn_hdr = """```python
from scapy.all import IP, TCP, UDP
def handle(pkt):
    if IP in pkt:
        proto = pkt[IP].proto
        ttl = pkt[IP].ttl
    if TCP in pkt:
        dport = pkt[TCP].dport
        sport = pkt[TCP].sport
        flags = pkt[TCP].flags
        win = pkt[TCP].window
    elif UDP in pkt:
        dport = pkt[UDP].dport
        sport = pkt[UDP].sport
    plen = len(pkt)
    # print(proto, ttl, sport, dport, flags, win, plen)
```"""

sn_flow = """```python
from scapy.all import IP, TCP, UDP
flows = {}
def key(pkt):
    if IP not in pkt: return None
    proto = 'TCP' if TCP in pkt else ('UDP' if UDP in pkt else str(pkt[IP].proto))
    s = pkt[IP].src; d = pkt[IP].dst
    sp = pkt[TCP].sport if TCP in pkt else (pkt[UDP].sport if UDP in pkt else 0)
    dp = pkt[TCP].dport if TCP in pkt else (pkt[UDP].dport if UDP in pkt else 0)
    return (s,d,sp,dp,proto)

def handle(pkt):
    k = key(pkt)
    if not k: return
    now = pkt.time
    x = flows.get(k)
    if x is None:
        flows[k] = x = dict(start=now, last=now, fwd=k, 
                            f_pkts=0, b_pkts=0, bytes=0, 
                            f_last=None, b_last=None, f_iat_sum=0.0, b_iat_sum=0.0)
    # Direction by comparing (src, sport) with flow key
    is_fwd = (IP in pkt and pkt[IP].src == k[0] and 
              ((TCP in pkt and pkt[TCP].sport == k[2]) or (UDP in pkt and pkt[UDP].sport == k[2]) or (not (TCP in pkt or UDP in pkt))))
    if is_fwd:
        x['f_pkts'] += 1
        if x['f_last'] is not None: x['f_iat_sum'] += now - x['f_last']
        x['f_last'] = now
    else:
        x['b_pkts'] += 1
        if x['b_last'] is not None: x['b_iat_sum'] += now - x['b_last']
        x['b_last'] = now
    x['bytes'] += len(pkt)
    x['last'] = now
    # Example derived fields
    dur = max(1e-6, x['last'] - x['start'])
    pkts_s = (x['f_pkts'] + x['b_pkts']) / dur
    bytes_s = x['bytes'] / dur
    f_iat_mean = (x['f_iat_sum']/max(1, x['f_pkts']-1)) if x['f_pkts']>1 else 0.0
    b_iat_mean = (x['b_iat_sum']/max(1, x['b_pkts']-1)) if x['b_pkts']>1 else 0.0
    # print(dur, pkts_s, bytes_s, f_iat_mean, b_iat_mean)
```"""

# Packet/header fields
mk('Source Port', True, True,
   'Direct from TCP/UDP header; trivial to read for each packet.', sn_hdr)
mk('Destination Port', True, True,
   'Direct from TCP/UDP header; trivial to read for each packet.', sn_hdr)
mk('Protocol (IP.proto / TCP/UDP)', True, True,
   'From IP header or by layer presence (TCP/UDP in pkt).', sn_hdr)
mk('IP.ttl', True, True,
   'From IPv4 header field; visible without extra privileges beyond sniff.', sn_hdr)
mk('TCP.window (Init_Win_bytes_forward/backward proxy)', True, True,
   'Available on TCP packets; can approximate initial window sizes when handshake seen.', sn_hdr)
mk('Packet Length (len(packet))', True, True,
   'Always available for captured frames; can be affected by offloads if not capturing on raw device.', sn_hdr)
mk('TCP Flags (per-packet)', True, True,
   'SYN/ACK/FIN/RST/PSH/URG/ECE/CWR are accessible via pkt[TCP].flags.', sn_hdr)

# Light per-flow aggregates
mk('Flow Duration (simple)', True, True,
   'First-seen to last-seen time for a 5‑tuple; O(1) updates per packet.', sn_flow)
mk('Total Fwd Packets', True, True,
   'Count packets in “forward” direction defined by first packet’s src/sport.', sn_flow)
mk('Total Backward Packets', True, True,
   'Count packets in the reverse direction for the same 5‑tuple.', sn_flow)
mk('Flow Packets/s (approx)', True, True,
   'Packets divided by flow duration; stable enough for short windows.', sn_flow)
mk('Flow Bytes/s (approx)', True, True,
   'Bytes divided by flow duration; CPU‑light; precise enough for IDS heuristics.', sn_flow)
mk('Fwd IAT Mean (simple)', True, True,
   'Maintain last timestamp per direction; average deltas online.', sn_flow)
mk('Bwd IAT Mean (simple)', True, True,
   'Same as Fwd IAT but for reverse direction.', sn_flow)

# Flag counts over the flow (CIC-style)
mk('SYN Flag Count', True, True,
   'Increment on TCP packets with SYN flag set.', """```python
# inside handle(pkt): if TCP in pkt and pkt[TCP].flags & 0x02: syn_cnt += 1
```""")
mk('ACK Flag Count', True, True,
   'Increment on TCP packets with ACK flag set.', """```python
# inside handle(pkt): if TCP in pkt and pkt[TCP].flags & 0x10: ack_cnt += 1
```""")
mk('FIN Flag Count', True, True,
   'Increment on TCP packets with FIN flag set.', """```python
# inside handle(pkt): if TCP in pkt and pkt[TCP].flags & 0x01: fin_cnt += 1
```""")
mk('RST Flag Count', True, True,
   'Increment on TCP packets with RST flag set.', """```python
# inside handle(pkt): if TCP in pkt and pkt[TCP].flags & 0x04: rst_cnt += 1
```""")
mk('PSH Flag Count', True, True,
   'Increment on TCP packets with PSH flag set.', """```python
# inside handle(pkt): if TCP in pkt and pkt[TCP].flags & 0x08: psh_cnt += 1
```""")
mk('URG Flag Count', True, True,
   'Increment on TCP packets with URG flag set.', """```python
# inside handle(pkt): if TCP in pkt and pkt[TCP].flags & 0x20: urg_cnt += 1
```""")
mk('ECE Flag Count', True, True,
   'Increment on TCP packets with ECE flag set.', """```python
# inside handle(pkt): if TCP in pkt and pkt[TCP].flags & 0x40: ece_cnt += 1
```""")
mk('CWR/CWE Flag Count', True, True,
   'Increment on TCP packets with CWR (a.k.a. CWE) flag set.', """```python
# inside handle(pkt): if TCP in pkt and pkt[TCP].flags & 0x80: cwr_cnt += 1
```""")

# Packet length summaries (window/flow)
mk('Min Packet Length (flow)', True, True,
   'Track running min of len(pkt) for the flow.', """```python
# inside handle(pkt): x['min_len'] = min(x.get('min_len', 1e9), len(pkt))
```""")
mk('Max Packet Length (flow)', True, True,
   'Track running max of len(pkt) for the flow.', """```python
# inside handle(pkt): x['max_len'] = max(x.get('max_len', 0), len(pkt))
```""")
mk('Average Packet Size (flow)', True, True,
   'Compute bytes/packets for the flow; matches CIC’s idea approximately.', """```python
# inside handle(pkt): avg_sz = x['bytes'] / max(1, x['f_pkts']+x['b_pkts'])
```""")
mk('Down/Up Ratio (approx)', True, True,
   'Use packet or byte counts per direction: (b_pkts or b_bytes) / (f_pkts or f_bytes).', """```python
# inside handle(pkt): ratio = x['b_pkts'] / max(1, x['f_pkts'])
```""")

scapy_df = pd.DataFrame(rows)
display(scapy_df)


### Notes for running the snippets on Linux/Pi (outside this notebook)

- Root privileges are typically required to sniff: run with `sudo` or grant CAP_NET_RAW. Example: `sudo python your_script.py`.
- Select the right interface (e.g., `iface="eth0"` or `wlan0`). Consider promiscuous mode if needed.
- NIC/kernel offloads (TSO/GRO/LRO) can affect observed lengths/timestamps. Prefer capturing on the raw interface or disable offloads when measuring sizes/rates.
- Raspberry Pi is capable of all listed features, but sustained high‑rate flows may require coarser time windows (e.g., 1–5s) to keep CPU usage low.
